In [ ]:
import os
import sys

In [ ]:
if os.path.abspath("../package") not in sys.path:
    sys.path.append(os.path.abspath("../package"))

In [ ]:
from package.environment import build_environment
from package.module import InverseDynamicsModel

In [ ]:
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader, random_split
from tensordict import TensorDict
from tensordict import MemoryMappedTensor as MemMap
from torchrl.envs import GymWrapper
from typing import Callable

In [ ]:
environment: GymWrapper = build_environment()
td: TensorDict = environment.rollout(max_steps=1000, break_when_any_done=False)

In [ ]:
def save_video_opencv(frames: torch.Tensor, out_path: str = "video.mp4", fps: int = 15):
    if frames.dtype != torch.uint8:
        frames: torch.Tensor = (frames.clamp(0, 1) * 255).to(torch.uint8)
    frames: np.ndarray = frames.permute(0, 2, 3, 1).cpu().numpy()
    height, width = frames.shape[1:3]
    fourcc: int = cv2.VideoWriter.fourcc(*"mp4v")
    vw: cv2.VideoWriter = cv2.VideoWriter(out_path, fourcc, fps, (width, height))
    for frame in frames:
        bgr: np.ndarray = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
        vw.write(bgr)
    vw.release()
    print(f"Saved in {out_path}")

In [ ]:
# save_video_opencv(td["observation"], fps=30)

In [ ]:
def progress(step: int, total: int, description: str = "") -> None:
    width: int = 40
    filled: int = int(width * step / total)
    bar: str = "█" * filled + "-" * (width - filled)
    sys.stdout.write(f"\r[{bar}] {step}/{total};" + " " + description)
    sys.stdout.flush()

In [ ]:
class InverseDynamicsDataset(Dataset):
    _AcceptedKeys: tuple[str, ...] = ("action", "observation", ("next", "observation"))

    def __init__(
            self,
            env_builder: Callable[[], GymWrapper],
            size: int,
            frames_per_session: int = 25,
            directory: str = "memmap",
            device: torch.device = torch.device("cpu"),
    ):
        assert (size % frames_per_session) == 0, f"{size=} must be a divided by {frames_per_session=}."
        self.env_builder: Callable[[], GymWrapper] = env_builder
        self.size: int = size
        self.directory: str = directory
        self.frames_per_session: int = frames_per_session
        self.device: torch.device = device
        self.x: MemMap | None = None
        self.y: MemMap | None = None
        self.initial()

    def init_storage(self, example: torch.Tensor, filename: str) -> MemMap:
        _, *tensor_shape = example.shape
        storage = MemMap.empty(shape=(self.size, *tensor_shape), dtype=example.dtype, filename=filename, existsok=True)
        return storage

    def initial(self) -> None:
        env: GymWrapper = self.env_builder()
        epochs: int = self.size // self.frames_per_session
        assert type(self.x) == type(self.y), "Something went wrong with storages."
        for epoch in range(epochs):
            progress(epoch + 1, epochs)
            tensordict: TensorDict = env.rollout(max_steps=self.frames_per_session, break_when_any_done=False)
            action, obs, next_obs = tensordict["action"], tensordict["observation"], tensordict["next", "observation"]
            x: torch.Tensor = torch.stack((obs, next_obs), dim=1).to(torch.uint8)
            y: torch.Tensor = action.to(torch.uint8)
            if self.x is None:
                self.x: MemMap = self.init_storage(x, filename=os.path.join(self.directory, "dset_x.memmap"))
                self.y: MemMap = self.init_storage(y, filename=os.path.join(self.directory, "dset_y.memmap"))
            assert isinstance(self.x, MemMap) and isinstance(self.y, MemMap), "Something went wrong."
            idx: slice = slice(epoch * self.frames_per_session, (epoch + 1) * self.frames_per_session)
            self.x[idx] = x
            self.y[idx] = y
        progress(epochs, epochs, description="Completed.")
        env.close()

    def __len__(self) -> int:
        return self.size

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        assert isinstance(self.x, MemMap) and isinstance(self.y, MemMap), "Something went wrong."
        return self.x[idx].to(self.device), self.y[idx].to(self.device)

In [ ]:
dset: InverseDynamicsDataset = InverseDynamicsDataset(env_builder=build_environment,
                                                      size=100000,
                                                      frames_per_session=25,
                                                      device="mps")

In [ ]:
train_dset, valid_dset = random_split(dset, (.75, .25))
valid_loader: DataLoader = DataLoader(valid_dset, 256, shuffle=False)
train_loader: DataLoader = DataLoader(train_dset, 256, shuffle=True)

In [ ]:
n_epochs: int = 1000
model = InverseDynamicsModel(n_actions=environment.action_space.n, scale=1.0 / 255.0).to(torch.device("mps"))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, eps=1e-5, weight_decay=1e-6)
loss_function = nn.CrossEntropyLoss()

In [ ]:
if os.path.exists("models/InverseDynamicsModel.pth"):
    model.load_state_dict(torch.load("models/InverseDynamicsModel.pth"))

In [ ]:
def eval_batch(network: nn.Module, batch: tuple[torch.Tensor, torch.Tensor], grad: bool = True):
    with torch.enable_grad() if grad else torch.no_grad():
        x, y = batch
        x1, x2, = torch.split(x, split_size_or_sections=1, dim=1)
        x1, x2 = x1.squeeze(dim=1), x2.squeeze(dim=1)
        probabilities: torch.Tensor = network(x1, x2).softmax(dim=-1)
        return x1, x2, y, probabilities

In [ ]:
from torch.optim import Optimizer
from torch.optim.lr_scheduler import LRScheduler

In [ ]:
def optim_step(loss_value: torch.Tensor, optim: Optimizer, lr_scheduler: LRScheduler | None = None) -> None:
    optim.zero_grad()
    loss_value.backward()
    optim.step()
    if lr_scheduler is not None:
        lr_scheduler.step()

In [ ]:
import torchmetrics.functional as tfm

In [ ]:
def get_metrics(a: torch.Tensor, b: torch.Tensor, num_classes: int) -> dict[str, int | float]:
    accuracy = tfm.accuracy(a, b, task="multiclass", num_classes=num_classes, average="macro")
    precision = tfm.precision(a, b, task="multiclass", num_classes=num_classes, average="macro")
    recall = tfm.recall(a, b, task="multiclass", num_classes=num_classes, average="macro")
    f1_score = tfm.f1_score(a, b, task="multiclass", num_classes=num_classes, average="macro")
    return dict(accuracy=accuracy, precision=precision, recall=recall, f1_score=f1_score)

In [ ]:
def pretty_print_dict(dictionary: dict[str, int | float]) -> str:
    return ", ".join([f"{key}={value:.4f}" for key, value in dictionary.items()])

In [ ]:
def train_iteration(network: nn.Module,
                    loss_fn: nn.Module,
                    num_epochs: int,
                    train_ldr: DataLoader,
                    valid_ldr: DataLoader):
    for epoch in range(num_epochs):
        # --------------------
        total_loss: float = 0.0
        for num, batch in enumerate(train_ldr):
            x1, x2, y, probabilities = eval_batch(network, batch, grad=True)
            loss: torch.Tensor = loss_fn(probabilities, y)
            optim_step(loss_value=loss, optim=optimizer)
            total_loss += loss.detach().item()
        description: str = f"Train Loss: {total_loss / len(train_ldr):.4f}"
        progress(epoch + 1, num_epochs, description=description)
        # --------------------
        total_loss: float = 0.0
        total_metrics: TensorDict | None = None
        for num, batch in enumerate(valid_ldr):
            x1, x2, y, probabilities = eval_batch(network, batch, grad=False)
            loss: torch.Tensor = loss_fn(probabilities, y)
            forecast: torch.Tensor = probabilities.argmax(dim=-1, keepdim=False)
            metrics: dict[str, torch.Tensor] = get_metrics(forecast, y, num_classes=4)
            total_loss += loss.detach().item()
            if total_metrics is None:
                total_metrics = TensorDict(metrics)
            else:
                total_metrics += TensorDict(metrics)
        assert isinstance(total_metrics, TensorDict), "Something went wrong."
        total_metrics: TensorDict = total_metrics / len(valid_ldr)
        description: str = f"Valid Loss: {total_loss / len(valid_ldr):.4f}, metrics: {pretty_print_dict(total_metrics.to_dict())}"
        progress(epoch + 1, num_epochs, description=description)

In [ ]:
train_iteration(network=model,
                loss_fn=loss_function,
                num_epochs=n_epochs,
                train_ldr=train_loader,
                valid_ldr=valid_loader)

In [ ]:
os.makedirs("models", exist_ok=True)
torch.save(model.state_dict(), "models/InverseDynamicsModel.pth")